In [1]:
using Pkg
Pkg.activate("..")
Pkg.resolve()
Pkg.instantiate()
Pkg.precompile()
import HybridDynamics as HD
import Plots as plt
using LinearAlgebra
plt.gr()

  Activating project at `c:\Users\david\Desktop\Program\VSCode\HybridDynamics.jl`
     Project No packages added to or removed from `C:\Users\david\Desktop\Program\VSCode\HybridDynamics.jl\Project.toml`
    Manifest No packages added to or removed from `C:\Users\david\Desktop\Program\VSCode\HybridDynamics.jl\Manifest.toml`


Plots.GRBackend()

TESTING ADAPTIVE LMMS WITH GENERAL SYSTEMS

In [2]:
function f_cont(x, t)
    # Continuous dynamics: [velocity, constant acceleration]
    return [x[2], -9.81]
end

f_cont (generic function with 1 method)

In [3]:
function h_guard(x)
    # Guard surface: triggers when position hits 0
    return x[1] 
end

h_guard (generic function with 1 method)

In [4]:
function reset_map(x, t=0.0)
    return [x[1], -0.8 * x[2]]
end
# Adding the second argument 't' allows the Variational code to call it,

reset_map (generic function with 2 methods)

In [5]:
# Solve Standard System
sys = HD.GeneralSystem(f_cont, h_guard, reset_map)
prob = HD.prob(sys, [10.0, 0.0], (0.0, 10.0))
sol = HD.solve(prob, HD.AdaptiveABM2())

HybridDynamics.GeneralSolution{Vector{Float64}}([0.0, 0.01, 0.02, 0.05, 0.14, 0.41000000000000003, 1.2200000000000002, 3.6500000000000004, 3.6500000000000004, 3.66, 3.67, 3.6999999999999997, 3.7899999999999996, 4.06, 4.869999999999999, 7.299999999999999, 10.0], [[10.0, 0.0], [9.9995095, -0.0981], [9.998038000000001, -0.1962], [9.987737500000001, -0.49050000000000005], [9.903862000000002, -1.3734000000000002], [9.175469500000002, -4.0221], [2.6993980000000004, -11.968200000000001], [-55.34686250000002, -35.80650000000001], [-55.34686250000002, 28.645200000000006], [-55.06090100000002, 28.547100000000007], [-54.77592050000002, 28.44900000000001], [-53.92686500000002, 28.15470000000001], [-51.43267250000002, 27.27180000000001], [-44.42686100000002, 24.623100000000008], [-27.70032050000001, 16.677000000000007], [-16.138745, -7.161299999999997], [-71.23170500000003, -33.648300000000006]], [3.6500000000000004], [9])

In [6]:
# Plot Standard System
t_plot, X_plot = HD.split_jumps(sol)
pos = [x[1] for x in X_plot]
vel = [x[2] for x in X_plot]

18-element Vector{Float64}:
   0.0
  -0.0981
  -0.1962
  -0.49050000000000005
  -1.3734000000000002
  -4.0221
 -11.968200000000001
 -35.80650000000001
 NaN
  28.645200000000006
  28.547100000000007
  28.44900000000001
  28.15470000000001
  27.27180000000001
  24.623100000000008
  16.677000000000007
  -7.161299999999997
 -33.648300000000006

In [7]:
p1 = plt.plot(t_plot, pos, label="Position", xlabel="Time (s)", ylabel="State", lw=2, color=:blue)
plt.plot!(p1, t_plot, vel, label="Velocity", lw=2, color=:red)
plt.title!("Standard System: Adaptive LMM")
display(p1)

print device already activated
print device already activated


VARIATIONAL EQUATION TEST WITH GENERAL SYSTEM

In [8]:
# Dynamics for [x, Φ]
#f_aug(U, t) = HD.variational_vector_field(f_cont, U, t)

#This is needed if you DO NOT use MagnusLeapfrog method.  

In [9]:
# Reset for [x, Φ]
function reset_aug(U)
    return HD.apply_variational_jump(U, f_cont, reset_map, h_guard, 0.0)
end

reset_aug (generic function with 1 method)

In [10]:
# Guard for augmented state (check guard on x component)
h_aug(U) = h_guard(U[:, 1])

h_aug (generic function with 1 method)

In [11]:
sys_aug = HD.GeneralSystem(f_cont, h_guard, reset_aug) #Change to f_aug if not using magnusLeapfrog
U0 = hcat([10.0, 0.0], Matrix{Float64}(I, 2, 2))
prob_aug = HD.prob(sys_aug, U0, (0.0, 10.0))
sol_aug = HD.solve(prob_aug, HD.MagnusLeapfrog())

┌ Info: Zeno contraction detected. count: 1
└ @ HybridDynamics C:\Users\david\Desktop\Program\VSCode\HybridDynamics.jl\src\Systems\General.jl:93


HybridDynamics.GeneralSolution{Matrix{Float64}}([0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.060000000000000005, 0.07, 0.08, 0.09  …  9.911958718453613, 9.921958718453613, 9.931958718453613, 9.941958718453613, 9.951958718453612, 9.961958718453612, 9.971958718453612, 9.981958718453612, 9.991958718453612, 10.0], [[10.0 1.0 0.0; 0.0 0.0 1.0], [9.9995095 1.0 0.01; -0.0981 0.0 1.0], [9.998038000000001 1.0 0.02; -0.1962 0.0 1.0], [9.9955855 1.0 0.03; -0.2943 0.0 1.0], [9.992152 1.0 0.04; -0.3924 0.0 1.0], [9.987737500000001 1.0 0.05; -0.49050000000000005 0.0 1.0], [9.982342000000001 1.0 0.060000000000000005; -0.5886 0.0 1.0], [9.975965500000001 1.0 0.07; -0.6867 0.0 1.0], [9.968608000000001 1.0 0.08; -0.7847999999999999 0.0 1.0], [9.9602695 1.0 0.09; -0.8828999999999999 0.0 1.0]  …  [-321.3196143926232 4.112320892868496 8.093737333191381; -79.39956323860973 0.508086478354085 0.9999999999999913], [-322.1141005250093 4.117401757652037 8.103737333191381; -79.49766323860973 0.508086478354085 0.99999999

In [12]:
# Plot Augmented System
t_plot_aug, U_plot = HD.split_jumps(sol_aug)

pos_a   = [U[1, 1] for U in U_plot]
vel_a   = [U[2, 1] for U in U_plot]
phi_11  = [U[1, 2] for U in U_plot]
phi_12  = [U[1, 3] for U in U_plot]
phi_21  = [U[2, 2] for U in U_plot]
phi_22  = [U[2, 3] for U in U_plot];

In [13]:
p2 = plt.plot(t_plot_aug, pos_a, label="Position", lw=2, color=:blue)
plt.plot!(p2, t_plot_aug, vel_a, label="Velocity", lw=2, color=:red)
plt.title!(p2, "Augmented System: State")

p3 = plt.plot(t_plot_aug, phi_11, label="∂x/∂x₀", lw=2, color=:purple)
plt.plot!(p3, t_plot_aug, phi_12, label="∂x/∂v₀", lw=2, color=:orange, linestyle=:dash)
plt.plot!(p3, t_plot_aug, phi_21, label="∂v/∂x₀", lw=2, color=:green)
plt.plot!(p3, t_plot_aug, phi_22, label="∂v/∂v₀", lw=2, color=:cyan, linestyle=:dash)
plt.title!(p3, "Augmented System: Sensitivity (Φ)")
plt.xlabel!(p3, "Time (s)")

final_plot = plt.plot(p2, p3, layout=(2,1), size=(800, 800))
display(final_plot)

print device already activated
print device already activated
print device already activated
